In [2]:
import os

In [3]:
pwd

'd:\\End-to-End-DL-Pipeline-for-Kidney-Disease-Classification-\\notebook'

In [4]:
os.chdir("../")

In [5]:
pwd

'd:\\End-to-End-DL-Pipeline-for-Kidney-Disease-Classification-'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
  root_dir: Path
  source_URL: str
  local_data_file_path: Path
  unzip_dir: Path

In [9]:
from src.constants.constants import *
from src.utils.utils import read_yaml, create_directories

In [11]:
class ConfigurationManager:
  def __init__(
    self,
    config_filepath = CONFIG_FILE_PATH,
    params_filepath = PARAMS_FILE_PATH):
    
    self.config = read_yaml(config_filepath)
    self.params = read_yaml(params_filepath)
    
    create_directories([self.config.artifacts_root])
    
  def get_data_ingestion_config(self) -> DataIngestionConfig:
    config = self.config.data_ingestion
    
    create_directories([config.root_dir])
    
    data_ingestion_config = DataIngestionConfig(
      root_dir=config.root_dir,
      source_URL=config.source_URL,
      local_data_file_path=config.local_data_file_path,
      unzip_dir=config.unzip_dir
    )
    return data_ingestion_config

In [13]:
import zipfile
import os
import sys
import gdown
from src.logger.logger import logger
from src.exception.exception import CustomException
from src.utils.utils import get_size

In [ ]:
class DataIngestion:
  def __init__(self, config: DataIngestionConfig):
    self.config = config
    
  def download_data(self):
    """Function to download data from google drive"""
    try:
      dataset_url = self.config.source_URL
      zip_download_dir = self.config.local_data_file_path
      os.makedirs(self.config.root_dir, exist_ok=True)
      logger.info(f"Downloading data from {dataset_url} to {zip_download_dir}")
      
      file_id = dataset_url.split('/')[-2]
      prefix_url = 'https://drive.google.com/uc?id='
      complete_url = prefix_url + file_id
      gdown.download(complete_url, zip_download_dir)
      logger.info(f'Downloaded data from {dataset_url} to {zip_download_dir}')
      
    except Exception as e:
      logger.error(f"Error downloading data from {dataset_url} to {zip_download_dir}: {e}")
      raise CustomException(e, sys)
    
  def unzip_data(self):
    """Function to unzip data"""
    try:
      unzip_dir = self.config.unzip_dir
      os.makedirs(unzip_dir, exist_ok=True)
      with zipfile.ZipFile(self.config.local_data_file_path, 'r') as zip_ref:
        zip_ref.extractall(unzip_dir)
      logger.info(f"Unzipped data from {self.config.local_data_file_path} to {unzip_dir}")
      
    except Exception as e:
      logger.error(f"Error unzipping data from {self.config.local_data_file_path} to {unzip_dir}: {e}")
      raise CustomException(e, sys)

In [ ]:
try:
  config = ConfigurationManager()
  data_ingestion_config = config.get_data_ingestion_config()
  data_ingestion = DataIngestion(config = data_ingestion_config)
  data_ingestion.download_data()
  data_ingestion.unzip_data()
  
except Exception as e:
  raise e